# B. SED - Spectral Energy Distribution

Spectral Energy Distribution for Key Objects. Spectral Profile of an object, compares the flux in all available filters (max is currently 10 [u,g,r,Vis,i,z,y,Y,J,H]) for all objects in the "Key Objects" dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path
import getpass

from lsst.rsp import get_tap_service, retrieve_query
from lsst.daf.butler import Butler, CollectionType
from astropy.table import Table, join
from astropy.coordinates import SkyCoord
from astropy import units as u

In [ ]:
my_username = getpass.getuser()
BASE = Path().resolve().parent.parent  # project root

## IF YOU CHANGE/ WORKING UNDER DIFFERENT FOLDER YOU MUST CHANGE THIS , KB_462 IS MINE
BASE = Path("/home/"+my_username+"/KB_462")
DATA = BASE / "data"

print("Working dir:", Path().resolve())

Input the final csv after completing Part A, Part 3

In [ ]:
#
Input = DATA / "Key_A.csv"
#

Option to toggle if figure are saved

In [ ]:
#SED PLOTS

filtered_cluster = [pd.read_csv(Input)]
n = len(filtered_cluster[0])

def lambdaVsFluxAlt(filtered_cluster,n):
    k = 0
    for cluster in filtered_cluster:
        k = k + 1
        # LSST filter wavelength ranges: https://coloringtheuniverse.netlify.app/rubin-lsst-camera-filters/
        # Euclid filter wavelength ranges : A) https://www.esa.int/Science_Exploration/Space_Science/Euclid/Euclid_s_instruments  ... B) https://euclid.caltech.edu/page/filters
        ulambda = 365*10
        glambda = 477*10
        rlambda = 621.5*10
        vislambda_sersic = 725*10 # effective wavelength, average of range [550 - 900] nm 
        ilambda = 754.4*10
        zlambda = 870*10
        ylambda = 1015*10
        ylambda_sersic = 1046*10 #
        jlambda_sersic = 1550*10 #
        hlambda_sersic = 1772*10 #

        # g = np.full(10, np.nan) 
        gals = range(len(cluster))   
        
        ufluxs = np.asarray(cluster[f"uFlux_LSST"].iloc[gals])
        gfluxs = np.asarray(cluster[f"gFlux_LSST"].iloc[gals])
        rfluxs = np.asarray(cluster[f"rFlux_LSST"].iloc[gals])
        ifluxs = np.asarray(cluster[f"iFlux_LSST"].iloc[gals])
        zfluxs = np.asarray(cluster[f"zFlux_LSST"].iloc[gals])
        yFluxs = np.asarray(cluster[f"yFlux_LSST"].iloc[gals])
        visfluxs_sersic = np.asarray(cluster[f"visFlux_sersic"].iloc[gals])
        yfluxs_sersic = np.asarray(cluster[f"yFlux_sersic"].iloc[gals])
        jfluxs_sersic = np.asarray(cluster[f"jFlux_sersic"].iloc[gals])
        hfluxs_sersic = np.asarray(cluster[f"hFlux_sersic"].iloc[gals])

        # SCALING: LSST in nJy and Euclid in uJy
        ufluxs = ufluxs/10**9
        gfluxs = gfluxs/10**9
        rfluxs = rfluxs/10**9
        ifluxs = ifluxs/10**9
        zfluxs = zfluxs/10**9
        yfluxs = yFluxs/10**9
        visfluxs_sersic = visfluxs_sersic/10**6
        yfluxs_sersic = yfluxs_sersic/10**6
        jfluxs_sersic = jfluxs_sersic/10**6
        hfluxs_sersic = hfluxs_sersic/10**6

        ra =  np.asarray(cluster[f"RA"].iloc[gals])
        dec =  np.asarray(cluster[f"DEC"].iloc[gals])
        tract = np.asarray(cluster[f"tract"].iloc[gals])
        patch = np.asarray(cluster[f"patch"].iloc[gals])
        objID = np.asarray(cluster[f"LSST_objID"].iloc[gals])

        
        for i in range(len(gals)):
            plt.figure()
            plt.plot(ulambda,ufluxs[i],'b.',label = 'u-flux-sersic') # LSST filters are all in sersic flux
            plt.plot(glambda,gfluxs[i],'g.',label = 'g-fl-sersicux-sersic')
            plt.plot(rlambda,rfluxs[i],'r.',label = 'r-flux-se-sersicrsic')
            plt.plot(ilambda,ifluxs[i],'c.',label = 'i-flux-sersic')
            plt.plot(zlambda,zfluxs[i],'m.',label = 'z-flux-sersic')
            plt.plot(ylambda,yfluxs[i],'y.',label = 'y-flux-sersic')
            plt.plot(vislambda_sersic,visfluxs_sersic[i], 'c*',label = 'vis-flux-sersic')
            plt.plot(ylambda_sersic,yfluxs_sersic[i],'y*',label = 'y-flux-sersic')
            plt.plot(jlambda_sersic,jfluxs_sersic[i],'*', label = 'j-flux-sersic', color = "brown")
            plt.plot(hlambda_sersic,hfluxs_sersic[i],'k*',label = 'h-flux-sersic')
            
            x = np.asarray([ulambda,glambda,rlambda, vislambda_sersic, ilambda, zlambda,ylambda, ylambda_sersic, jlambda_sersic, hlambda_sersic])
            y = np.asarray([ufluxs[i],gfluxs[i],rfluxs[i],ifluxs[i],visfluxs_sersic[i],zfluxs[i],yfluxs[i],yfluxs_sersic[i],jfluxs_sersic[i],hfluxs_sersic[i]])
            nan_int = np.where(~np.isnan(y))[0]
            x = x[nan_int]
            y = y[nan_int]
            p = np.polyfit(x, y, 4)      
            y_fit = np.polyval(p, x)
            plt.plot(x, y_fit, 'k-')
            plt.legend(loc = 'best', fontsize = 'x-small')
            plt.title(f'LSST ID: [{objID[i]}], [{tract[i]},{patch[i]}], [{ra[i]},{dec[i]}]')
            plt.xlabel('Wavelength [A]')
            plt.ylabel('Flux [Jy]')

            # TOGGLE TO SAVE FIGURES 
            (DATA / "plots" / "SED").mkdir(parents=True, exist_ok=True)
            plt.savefig(DATA / "plots" / "SED" / f"{int(objID[i])}_plot.png",dpi=300, bbox_inches="tight")
            #
            
    return{'ufluxs':ufluxs, 'ufluxs':ufluxs}

Excpeted Warning: RuntimeWarning: More than 20 figures have been opened

In [ ]:
lambdaVsFluxAlt(filtered_cluster,n)

## Part B Done

## Proceed to Part C Custom Coadds